In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

import os

figure_dir = "figures/revision/figure-3"
figure_dir_2 = "figures/revision/supplement"
setup_plotting(figure_dir, display_dpi=300, save_dpi=300)
setup_plotting(figure_dir_2, display_dpi=300, save_dpi=300)

import matplotlib.pyplot as plt
import nichepca as npc
import numpy as np
import pandas as pd
import scanpy as sc
import scipy
import seaborn as sns

from spatial_tcr.distances import distance_matrix
from spatial_tcr.nhood_analysis import (
    compute_nhood_composition,
    compute_nhood_expression,
)
from spatial_tcr.stats import (
    wilcoxon_significance,
)
from spatial_tcr.tcr import get_tcr_genes

## Load data

In [ ]:
data_dir = "data/xenium/processed"
path = os.path.join(data_dir, "08.1-kidney_tcr_clonal_clusters.h5ad")
adata = sc.read_h5ad(path)
adata = adata[adata.obs["condition"] == "ANCA"].copy()
adata.X = adata.layers["counts"].copy()
adata

In [ ]:
cc_key = "avbv_cluster_filtered"

In [ ]:
adata.obs["is_avbv_cluster"] = adata.obs[cc_key].notna().astype(str)
adata.obs.loc[adata.obs[cc_key].isna(), "is_avbv_cluster"] = None


# prepare other T cell column that excludes all T cells in clusters
adata.obs["ct_outside_cluster"] = adata.obs["cell_type_l2"].copy()
adata.obs.loc[adata.obs[cc_key].notna(), "ct_outside_cluster"] = None

## Construct neighborhood graph

In [ ]:
npc.gc.construct_multi_sample_graph(
    adata,
    sample_key="sample",
    obsm_key="spatial",
    radius=25,
    remove_self_loops=True,
    verbose=False,
)

## Compute cell type composition

In [ ]:
ad_nhood_tclonal = compute_nhood_composition(
    adata, obs_key="is_avbv_cluster", comp_key="cell_type_l2"
)
ad_nhood_not_tclonal = compute_nhood_composition(
    adata, obs_key="ct_outside_cluster", comp_key="cell_type_l2"
)
tcell_keys = [
    "CD4+",
    # "TFH",
    "Tregs",
    "MAIT",
    "CD8+",
    "NKT-like",
]
ad_nhood_tcells_not_clonal = ad_nhood_not_tclonal[
    ad_nhood_not_tclonal.obs["ct_outside_cluster"].isin(tcell_keys)
].copy()

In [ ]:
# merge the expression
var_names = ad_nhood_tclonal.var_names
ad_merged = sc.AnnData(
    X=np.concatenate(
        [
            ad_nhood_tclonal.X,
            ad_nhood_tcells_not_clonal[:, var_names].X,
        ],
        axis=0,
    ),
    var=ad_nhood_tclonal.var,
    obs=pd.concat([ad_nhood_tclonal.obs, ad_nhood_tcells_not_clonal.obs], axis=0),
)
ad_merged.obs["group"] = "cluster\nnhood."
ad_merged.obs.loc[ad_nhood_tcells_not_clonal.obs_names, "group"] = "non-cluster\nnhood."

In [ ]:
ct_comp_merged = ad_merged.to_df()
ct_comp_merged = pd.concat([ct_comp_merged, ad_merged.obs[["group"]]], axis=1)

In [ ]:
from statsmodels.stats.multitest import multipletests

group_key = "group"

ct_comp_merged_normalized = ct_comp_merged.copy()
ct_cols = [c for c in ct_comp_merged_normalized.columns if c != group_key]
ct_comp_merged_normalized[ct_cols] = ct_comp_merged_normalized[ct_cols].div(
    ct_comp_merged_normalized[ct_cols].sum(axis=1), axis=0
)

p_values = {}
for ct in ct_cols:
    p_values[ct] = wilcoxon_significance(
        ct_comp_merged_normalized, ct, group_key=group_key
    )

# perform multiple testing correction
p_values_corrected = multipletests(list(p_values.values()), method="fdr_bh")[1]
p_values = dict(zip(p_values.keys(), p_values_corrected))


mean_comp = ct_comp_merged_normalized.groupby(group_key).mean()
ratios = mean_comp.iloc[0] / mean_comp.iloc[1]
ratios = ratios.sort_values(ascending=True)
mean_comp = mean_comp[ratios.index]
# mean_comp.index = ["TFH\nCXCL13+", "TFH\nCXCL13-"]
mean_comp

In [ ]:
sns.set_theme(style="ticks", context="paper")

agg_comp_per_group_norm = mean_comp[mean_comp.columns[-10::]]
agg_comp_per_group_norm["other"] = 1 - agg_comp_per_group_norm.sum(axis=1)
# move other columns to the front
agg_comp_per_group_norm = agg_comp_per_group_norm[
    ["other"] + list(agg_comp_per_group_norm.columns[:-1])
]

celltypes = agg_comp_per_group_norm.columns
palette = sns.color_palette("tab10", len(celltypes))
palette = {ct: palette[i] for i, ct in enumerate(celltypes[::-1])}
palette["other"] = "lightgray"

fig, ax = plt.subplots(figsize=(1.5, 5))
agg_comp_per_group_norm.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[palette[ct] for ct in agg_comp_per_group_norm.columns],
    linewidth=0,
)

ax.set_title("T cell neighborhood composition")
ax.set_ylabel("Proportion")
ax.set_xlabel("")
# ax.get_legend().remove()
# ax.legend(
#     bbox_to_anchor=(0.5, -0.2),
#     loc="upper center",
#     ncol=7,
#     bbox_transform=ax.transAxes,
#     frameon=False,
# )
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], bbox_to_anchor=(1, 1), frameon=False, ncol=1)
sns.despine(ax=ax, right=True, top=True)

# set x-axis labels to ANCA and Control
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha="center")
plt.show()
# plt.tight_layout()

In [ ]:
nhood_ct_comp_melted = pd.melt(
    mean_comp.reset_index(),
    id_vars="group",
    var_name="cell_type",
    value_name="proportion",
)
nhood_ct_comp_melted.columns = ["group", "cell_type", "prop"]

In [ ]:
adata.obs.cell_type_l2.value_counts()

In [ ]:
sns.set_theme(style="ticks", context="paper")

relevant_cts = mean_comp.columns[::-1][:11]

relevant_cts = ["B", "MDC", "cDC", "pDC"]

scaler = 0.58
for ct in relevant_cts:
    colors = sns.color_palette("Greys", n_colors=2)[::-1]

    plot_df = nhood_ct_comp_melted[nhood_ct_comp_melted["cell_type"] == ct]

    plot_df.loc[:, "prop"] = plot_df["prop"] * 100

    fig, ax = plt.subplots(figsize=(1.5 * scaler, 5 * scaler))
    sns.barplot(
        x="group",
        y="prop",
        hue="group",
        data=plot_df.iloc[::-1],
        ax=ax,
        palette=colors,
        width=0.6,
    )
    ax.set_title(f"{ct}", fontsize=8)
    ax.set_xlabel("")
    ax.set_ylabel("proportion [%]", fontsize=8)

    # Add significance bar and star
    # Convert p-value to star notation
    # n_cells_clonal = ad_nhood_tclonal.obsm["ct_counts"].sum(axis=1).sum()
    # ct_cells_clonal = ad_nhood_tclonal.obsm["ct_counts"][ct].sum()
    # other_cells_clonal = n_cells_clonal - ct_cells_clonal

    # n_cells_nonclonal = ad_nhood_not_tclonal.obsm["ct_counts"].sum(axis=1).sum()
    # ct_cells_nonclonal = ad_nhood_not_tclonal.obsm["ct_counts"][ct].sum()
    # other_cells_nonclonal = n_cells_nonclonal - ct_cells_nonclonal
    # p_value = count_test(
    #     ct_cells_clonal, other_cells_clonal, ct_cells_nonclonal, other_cells_nonclonal
    # )
    p_value = p_values[ct]
    print(f"{ct}: {p_value}")
    if p_value < 0.001:
        stars = "***"
    elif p_value < 0.01:
        stars = "**"
    elif p_value < 0.05:
        stars = "*"
    else:
        stars = "ns"
    # Adjust the y-offset based on your data range:
    y_max = np.max(plot_df["prop"].values)
    y = y_max + 0.05 * (y_max)  # starting height of the line
    h = 0.02 * (y_max)  # height of the significance bar

    x1, x2 = 0, 1
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1.0, c="k")
    ax.text(
        (x1 + x2) * 0.5, y + h, stars, ha="center", va="bottom", color="k", fontsize=6
    )

    # Add a bit more space at the top of the plot for significance annotation
    ax.set_ylim(top=y + h + 0.05 * y_max, bottom=0)
    ax.tick_params(axis="both", labelsize=6)

    sns.despine()
    # plt.tight_layout()
    plt.savefig(
        os.path.join(figure_dir, f"cc_nhood_comp_{ct}.pdf"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

## Analyze distance to APCs and compare for other t cells

In [ ]:
# create col with t in clonal cluster, t outside clonal cluster and other cell type labels
adata.obs["tmp"] = adata.obs["cell_type_l2"].astype(str)
t_mask = adata.obs["cell_type_l2"].isin(tcell_keys)
adata.obs.loc[t_mask, "tmp"] = "other abT"
adata.obs.loc[adata.obs[cc_key].notna(), "tmp"] = "clustered abT"
# adata.obs["tmp"].value_counts()

In [ ]:
dist_mat, dist_mat_per_sample, dists_combined = distance_matrix(
    adata, obs_key="tmp", sample_key="cc"
)

In [ ]:
apcs = ["B", "pDC", "cDC", "MDC"]
# dist_mat_relevant = dist_mat.loc[
#     ["other abT", "clustered abT"], apcs + ["other abT", "clustered abT"]
# ]
dist_mat_relevant = dist_mat.loc[["other abT", "clustered abT"]]
dist_mat_relevant

### Bar plot

In [ ]:
df = dist_mat_relevant.T.reset_index()
df["ratio"] = df["clustered abT"] / df["other abT"]
df = df.sort_values("ratio", ascending=True)
df.drop(columns=["ratio"], inplace=True)
display(df.head())


df = df.melt(id_vars="index", var_name="class")
mask = df["index"].isin(("other abT", "clustered abT"))
df = df[~mask].reset_index(drop=True)
df.head()

In [ ]:
sns.set_theme(style="ticks", context="paper")
plt.figure(figsize=(6, 5))
mask = df["index"].isin(["cDC", "B", "MDC", "pDC"])
sns.barplot(
    data=df[mask],
    x="index",
    y="value",
    hue="class",
    # palette=["#ff8c8c", "#7fbfff"],
    # alpha=0.8,
    width=0.8,
)

plt.xticks(rotation=0, ha="center")
plt.xlabel("Cell Type")
plt.ylabel("Distance [µm]")
# plt.title("Distance Between Cell Types", pad=20)
plt.legend(
    title="T cell group", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False
)
sns.despine()
plt.tight_layout()

### Boxplot variant

In [ ]:
df = dists_combined.drop(columns=["sample"])
# df["ratio"] = df["clustered abT"] / df["other abT"]
# df = df.sort_values("ratio", ascending=True)
# df.drop(columns=["ratio"], inplace=True)
df = df.melt(id_vars="cell_type", var_name="other_cell_type", value_name="value")
df = df[df["cell_type"].isin(["other abT", "clustered abT"])]
df = df[~df["other_cell_type"].isin(["other abT", "clustered abT"])]
df.dropna(inplace=True)
df

In [ ]:
sns.set_theme(style="ticks", context="paper")
plt.figure(figsize=(6, 5))
mask = df["other_cell_type"].isin(["cDC", "B", "MDC", "pDC"])
sns.boxplot(
    data=df[mask],
    x="other_cell_type",
    y="value",
    hue="cell_type",
    # palette=["#ff8c8c", "#7fbfff"],
    # alpha=0.8,
    width=0.8,
    order=["cDC", "B", "MDC", "pDC"],
    fliersize=1,
    flierprops={"marker": ".", "alpha": 0.3},
)

plt.xticks(rotation=0, ha="center")
plt.xlabel("Cell Type")
plt.ylabel("Distance [µm]")
# plt.title("Distance Between Cell Types", pad=20)
plt.legend(
    title="T cell group", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False
)
sns.despine()
plt.tight_layout()

### Scatterplot

In [ ]:
df = dist_mat_relevant.T.reset_index()
mask = df["index"].isin(["clustered abT", "other abT"])
df = df[~mask]

# mask = df["index"].isin(["cDC", "B", "MDC", "pDC"])
# df = df[mask]
palette = sns.color_palette("tab10")


# def mapper(x):
#     if x not in ["cDC", "B", "MDC", "pDC"]:
#         return "other"
#     else:
#         return x
relevant_cts = ["cDC", "B", "MDC", "pDC"]

# df["index"] = df["index"].astype(str).map(mapper)
color_map = dict(zip(relevant_cts, palette[: len(relevant_cts)]))
for ct in df["index"].unique():
    if ct not in color_map:
        color_map[ct] = "lightgray"


sns.set_theme(style="ticks", context="paper")
scaler = 0.75
fig, ax = plt.subplots(figsize=(5 * scaler, 5 * scaler))

ct_ratios = adata.obs["tmp"].value_counts(normalize=True)
# Scale the sizes based on cell type ratios, multiplied by 5000 to make them visible
max_size = 400
min_size = 20
max_ratio = max(ct_ratios.values)
min_ratio = min(ct_ratios.values)
print(max_ratio, min_ratio)
sizes = [
    (ct_ratios[ct] / max_ratio) * (max_size - min_size) + min_size for ct in df["index"]
]
print(sizes)
sizes = [ct_ratios[ct] for ct in df["index"]]

sns.scatterplot(
    df,
    x="other abT",
    y="clustered abT",
    hue="index",
    ax=ax,
    size=sizes,
    sizes=(20, 200),
    # alpha=0.8,
    # legend="brief",
    palette=color_map,
)

# Convert to log scale
ax.set_xscale("log")
ax.set_yscale("log")


# Add diagonal line
ax.set_xlim(0, 225)
ax.set_ylim(0, 225)

lims = [
    np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
    np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
]
ax.plot(lims, lims, "k--", alpha=0.5, zorder=0)  # black dashed line
ax.set_aspect("equal")
ax.set_xlim(lims)
ax.set_ylim(lims)

ax.set_xlabel("Distance to other abT [µm]", fontsize=8)
ax.set_ylabel("Distance to clustered abT [µm]", fontsize=8)
ax.set_title("Distance comparison", fontsize=8)
ax.tick_params(axis="both", labelsize=6)

# Get the legend handles and labels
handles, labels = ax.get_legend_handles_labels()
print(handles)
print(labels)

# Create a new legend with only the color markers
new_handles = []
new_labels = []
seen_other = False
for h, l in zip(handles, labels):
    if l in relevant_cts:
        new_handles.append(h)
        new_labels.append(l)
    elif not seen_other:
        new_handles.append(h)
        new_labels.append("other")
        seen_other = True

# also add the size legend part
for h, l in zip(handles, labels):
    try:
        float(l)
        new_handles.append(h)
        new_labels.append(l)
    except (ValueError, TypeError):
        pass

legend = ax.legend(
    new_handles,  # Only keep the color markers
    new_labels,  # Only keep the labels
    title="Cell type",
    bbox_to_anchor=(1.01, 1),
    loc="upper left",
    frameon=False,
    fontsize=6,
    title_fontsize=6,
)
sns.despine()
plt.tight_layout()
plt.savefig(
    os.path.join(figure_dir, "distance_comparison.pdf"), dpi=300, bbox_inches="tight"
)
plt.show()

## Compute neighborhood expression

In [ ]:
ad_nhood_tclonal = compute_nhood_expression(
    adata, obs_key="is_avbv_cluster", construct_graph=False, layer="counts"
)
ad_nhood_not_tclonal = compute_nhood_expression(
    adata, obs_key="ct_outside_cluster", construct_graph=False, layer="counts"
)
tcell_keys = [
    "CD4+",
    # "TFH",
    "Tregs",
    "MAIT",
    "CD8+",
    "NKT-like",
]
print(adata[adata.obs["cell_type_l2"].isin(tcell_keys)].shape[0])
ad_nhood_tcells_not_clonal = ad_nhood_not_tclonal[
    ad_nhood_not_tclonal.obs["ct_outside_cluster"].isin(tcell_keys)
].copy()

In [ ]:
ad_nhood_tclonal.X.sum(axis=1).min()

In [ ]:
ad_nhood_tclonal.var_names

In [ ]:
# merge the expression
var_names = ad_nhood_tclonal.var_names
ad_merged = sc.AnnData(
    X=scipy.sparse.vstack(
        [ad_nhood_tclonal.X, ad_nhood_tcells_not_clonal[:, var_names].X]
    ),
    var=ad_nhood_tclonal.var,
    obs=pd.concat([ad_nhood_tclonal.obs, ad_nhood_tcells_not_clonal.obs], axis=0),
)
tv_genes = get_tcr_genes(ad_merged)[-1]
ad_merged = ad_merged[:, [g for g in ad_merged.var_names if g not in tv_genes]].copy()

ad_merged.obs["group"] = "cluster\nnhood."
ad_merged.obs.loc[ad_nhood_tcells_not_clonal.obs_names, "group"] = "non-cluster\nnhood."

sc.pp.normalize_total(ad_merged)
sc.pp.log1p(ad_merged)

In [ ]:
sc.tl.rank_genes_groups(ad_merged, groupby="group", method="wilcoxon")
sc.pl.rank_genes_groups(ad_merged, n_genes=25, show=False)
plt.show()

In [ ]:
import decoupler as dc

# get dataframe with ranked genes
ranked_genes = sc.get.rank_genes_groups_df(ad_merged, group="cluster\nnhood.")
ranked_genes["pvals_adj"] = ranked_genes["pvals_adj"].clip(lower=1e-300, upper=1)

sns.set_theme(style="ticks", context="paper")
fig, ax = plt.subplots(1, 1, figsize=(7, 5))
dc.pl.volcano(
    ranked_genes.set_index("names"),
    x="logfoldchanges",
    y="pvals_adj",
    thr_stat=0.5,
    thr_sign=1e-10,
    ax=ax,
    top=30,
)
ax.set_xlabel("log2(fold Change)")
ax.set_title(
    "differentially expressed genes in clustered vs not clustered T cell neighborhoods"
)
sns.despine()
plt.savefig(
    os.path.join(figure_dir_2, "volcano_clustered_nhoods_vs_t_nhoods.pdf"),
    dpi=300,
    bbox_inches="tight",
    transparent=True,
)
plt.show()

In [ ]:
import gseapy as gp

# NOTE: To speed up, use gp.prerank instead with your own ranked list.
res = gp.gsea(
    data=ad_merged.to_df().T,  # row -> genes, column-> samples
    gene_sets="GO_Biological_Process_2023",
    cls=ad_merged.obs.group,
    permutation_num=1000,
    permutation_type="phenotype",
    outdir=None,
    method="s2n",  # signal_to_noise
    threads=16,
)

In [ ]:
for _, row in list(res.res2d.iterrows())[0:10]:
    print(f"{row['Term']}: {row['Lead_genes']}")

In [ ]:
plot_df = res.res2d.copy()
plot_df["Term"] = plot_df["Term"].str.split(r"\(").str[0]
plot_df = plot_df.loc[plot_df["FDR q-val"] < 0.05]
plot_df = plot_df.head(10)

# plot as barplot
sns.set_theme(style="ticks", context="paper")
fig, ax = plt.subplots(1, 1, figsize=(2, 5))

sig_col = "FDR q-val"

sns.barplot(x="NES", y="Term", hue=sig_col, data=plot_df, ax=ax, palette="Reds_r")
ax.get_legend().remove()

norm = plt.Normalize(plot_df[sig_col].min(), plot_df[sig_col].max())
sm = plt.cm.ScalarMappable(cmap="Reds_r", norm=norm)
cbar = plt.colorbar(sm, ax=ax, aspect=5, shrink=0.3, label=sig_col)

ax.set_title("GSEA on clonal clustered vs other T cell neighborhoods")
ax.set_xlabel("Normalized Enrichment Score")
ax.set_ylabel("")
sns.despine()

plt.savefig(
    os.path.join(figure_dir, "gsea_clonal_vs_other_t_nhoods.pdf"),
    dpi=300,
    bbox_inches="tight",
)

plt.show()